# AI Laboratory — Bayesian Networks and Autoregressive Language Models

## Submission Notebook

This notebook contains the implementations, conditional probability tables, generation experiments, normalization tests, and first-order/second-order comparison.


## Dataset and model specification

The six-sentence dataset is the one supplied in the laboratory. Text is lower-cased, split into word tokens, and augmented with `<START>` and `<END>`.

First-order model: `P(X_t | X_{t-1})`

Second-order model: `P(X_t | X_{t-2}, X_{t-1})`

In [ ]:
from collections import defaultdict, Counter
import random

raw_sentences = [
    "the cat sat on the mat",
    "the cat sat on the rug",
    "the dog sat on the mat",
    "the dog ran to the park",
    "the cat ran to the park",
    "the dog sat on the rug",
]

sentences = [s.lower().split() for s in raw_sentences]

def add_boundaries(tokens):
    return ["<START>"] + tokens + ["<END>"]

In [ ]:
def build_first_order(data):
    counts = defaultdict(Counter)
    for tokens in data:
        seq = add_boundaries(tokens)
        for previous, current in zip(seq, seq[1:]):
            counts[previous][current] += 1

    probabilities = {}
    for previous, next_counts in counts.items():
        total = sum(next_counts.values())
        probabilities[previous] = {
            word: count / total
            for word, count in sorted(next_counts.items())
        }
    return counts, probabilities

first_counts, first_probabilities = build_first_order(sentences)

for word in sorted(first_counts):
    print(word, dict(first_counts[word]))

In [ ]:
def predict_next(probabilities, previous):
    distribution = probabilities.get(previous)
    if not distribution:
        return None
    return max(distribution, key=distribution.get)

def sample_next(probabilities, previous, rng):
    distribution = probabilities.get(previous)
    if not distribution:
        return None
    words, weights = zip(*distribution.items())
    return rng.choices(words, weights=weights, k=1)[0]

def generate_first_order(probabilities, rng, greedy=False, max_len=30):
    previous = "<START>"
    output = []
    for _ in range(max_len):
        current = (predict_next(probabilities, previous)
                   if greedy else sample_next(probabilities, previous, rng))
        if current is None or current == "<END>":
            break
        output.append(current)
        previous = current
    return " ".join(output)

## First-order conditional probability tables

In [ ]:
for word in ["the", "cat", "dog", "sat", "ran"]:
    print(f"P(next | {word}) =", first_probabilities.get(word, {}))

## Next-word predictions

In [ ]:
for context in ["<START>", "the", "cat", "dog", "sat", "ran", "on"]:
    print(f"{context:>8} -> {predict_next(first_probabilities, context)}")

## Twenty sampled first-order sentences

In [ ]:
rng = random.Random(42)
generated_20 = [
    generate_first_order(first_probabilities, rng, greedy=False)
    for _ in range(20)
]
for i, sentence in enumerate(generated_20, 1):
    print(f"{i:2}. {sentence}")

## Greedy versus sampling

In [ ]:
rng_sampling = random.Random(42)
rng_greedy = random.Random(42)

sampling_sentences = [
    generate_first_order(first_probabilities, rng_sampling, greedy=False)
    for _ in range(5)
]
greedy_sentences = [
    generate_first_order(first_probabilities, rng_greedy, greedy=True)
    for _ in range(5)
]

print("Sampling:")
for s in sampling_sentences:
    print("-", s)

print("\nGreedy:")
for s in greedy_sentences:
    print("-", s)

## Probability normalization test

In [ ]:
for word, distribution in first_probabilities.items():
    total = sum(distribution.values())
    print(word, total)
    assert abs(total - 1.0) < 1e-12

print("All first-order distributions sum to 1.")

## Second-order model

In [ ]:
def build_second_order(data):
    counts = defaultdict(Counter)
    for tokens in data:
        seq = ["<START>", "<START>"] + tokens + ["<END>"]
        for first, second, current in zip(seq, seq[1:], seq[2:]):
            counts[(first, second)][current] += 1

    probabilities = {}
    for context, next_counts in counts.items():
        total = sum(next_counts.values())
        probabilities[context] = {
            word: count / total
            for word, count in sorted(next_counts.items())
        }
    return counts, probabilities

second_counts, second_probabilities = build_second_order(sentences)

def predict_next_second(probabilities, context):
    distribution = probabilities.get(context)
    if not distribution:
        return None
    return max(distribution, key=distribution.get)

def sample_next_second(probabilities, context, rng):
    distribution = probabilities.get(context)
    if not distribution:
        return None
    words, weights = zip(*distribution.items())
    return rng.choices(words, weights=weights, k=1)[0]

def generate_second_order(probabilities, rng, greedy=False, max_len=30):
    first, second = "<START>", "<START>"
    output = []
    for _ in range(max_len):
        context = (first, second)
        current = (predict_next_second(probabilities, context)
                   if greedy else sample_next_second(probabilities, context, rng))
        if current is None or current == "<END>":
            break
        output.append(current)
        first, second = second, current
    return " ".join(output)

## Selected second-order conditional probability tables

In [ ]:
contexts = [
    ("<START>", "the"),
    ("the", "cat"),
    ("the", "dog"),
    ("cat", "sat"),
    ("dog", "sat"),
    ("dog", "ran"),
    ("sat", "on"),
]

for context in contexts:
    print(f"P(next | {context}) =", second_probabilities.get(context, {}))

## First-order versus second-order comparison

In [ ]:
rng_first = random.Random(42)
rng_second = random.Random(42)

second_greedy = [
    generate_second_order(second_probabilities, rng_first, greedy=True)
    for _ in range(5)
]
second_sampling = [
    generate_second_order(second_probabilities, rng_second, greedy=False)
    for _ in range(5)
]

print("Second-order greedy:")
for s in second_greedy:
    print("-", s)

print("\nSecond-order sampling:")
for s in second_sampling:
    print("-", s)

print("\nFirst-order non-zero parameters:", sum(len(d) for d in first_probabilities.values()))
print("Second-order non-zero parameters:", sum(len(d) for d in second_probabilities.values()))

## Final checks

In [ ]:
for distribution in first_probabilities.values():
    assert abs(sum(distribution.values()) - 1.0) < 1e-12

for distribution in second_probabilities.values():
    assert abs(sum(distribution.values()) - 1.0) < 1e-12

# Actual supplied dataset:
# P(cat | the) = 3/12 and P(dog | the) = 3/12.
assert first_counts["the"]["cat"] == 3
assert first_counts["the"]["dog"] == 3
assert first_counts["the"]["mat"] == 2
assert first_counts["the"]["rug"] == 2
assert first_counts["the"]["park"] == 2
assert abs(first_probabilities["the"]["cat"] - 0.25) < 1e-12
assert abs(first_probabilities["the"]["dog"] - 0.25) < 1e-12
assert len(generated_20) == 20

print("All probability-model checks passed.")